# URL Ranking System V2 (Interview Ready)

Features:
- SQLite Database
- 30 URLs
- Search Index
- TF-IDF Ranking
- Redis-like Cache with TTL
- Ranking Explanation
- Analytics Dashboard
- Click Simulation
- Monitoring Dashboard
- Architecture Visualization


## Section 1 - Architecture Visualization

In [ ]:

!pip -q install graphviz scikit-learn

from graphviz import Digraph

g = Digraph()
g.node('A','User')
g.node('B','Redis Cache')
g.node('C','Query Processor')
g.node('D','Search Index')
g.node('E','Ranking Engine')
g.node('F','Database')
g.node('G','Analytics')

g.edge('A','B')
g.edge('B','C')
g.edge('C','D')
g.edge('D','E')
g.edge('E','F')
g.edge('E','G')

g


## Section 2 - Imports

In [ ]:

import sqlite3
import pandas as pd
import numpy as np
import math
import time
from datetime import datetime
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## Section 3 - Database Setup

In [ ]:

conn = sqlite3.connect('search_engine.db')
cur = conn.cursor()

cur.executescript('''
DROP TABLE IF EXISTS urls;
DROP TABLE IF EXISTS query_logs;
DROP TABLE IF EXISTS click_logs;
DROP TABLE IF EXISTS search_index;

CREATE TABLE urls(
id INTEGER PRIMARY KEY,
url TEXT,
title TEXT,
keywords TEXT,
category TEXT,
clicks INTEGER,
backlinks INTEGER,
freshness INTEGER
);

CREATE TABLE query_logs(
id INTEGER PRIMARY KEY AUTOINCREMENT,
query TEXT,
timestamp TEXT
);

CREATE TABLE click_logs(
id INTEGER PRIMARY KEY AUTOINCREMENT,
url TEXT,
timestamp TEXT
);

CREATE TABLE search_index(
keyword TEXT,
url_id INTEGER
);
''')

conn.commit()
print("Database Created")


## Section 4 - Simulated Crawler Dataset (30 URLs)

In [ ]:

# Simulated crawler output

urls=[
('https://python.org/tutorial','Python Tutorial','python tutorial beginner coding','Programming',1200,500,95),
('https://realpython.com','Real Python Guide','python guide coding programming','Programming',1500,700,90),
('https://w3schools.com/python','Learn Python','python tutorial coding','Programming',1000,350,88),
('https://geeksforgeeks.org/python','Python Programming','python coding tutorial','Programming',1300,600,92),
('https://freecodecamp.org/python','Complete Python Course','python course tutorial','Programming',1700,800,91),
('https://tensorflow.org/tutorials','TensorFlow Tutorials','machine learning ai python','AI',2000,1200,96),
('https://pytorch.org/tutorials','PyTorch Tutorials','machine learning deep learning python','AI',1800,1100,94),
('https://kaggle.com/learn','Kaggle Learn','data science machine learning','AI',1600,900,93),
('https://towardsdatascience.com','Data Science Articles','data science ai machine learning','AI',1750,950,89),
('https://scikit-learn.org','Scikit Learn','machine learning python','AI',1550,850,92),
('https://aws.amazon.com','AWS Cloud','cloud aws infrastructure','Cloud',2100,1500,97),
('https://azure.microsoft.com','Azure Cloud','cloud azure infrastructure','Cloud',1800,1200,95),
('https://cloud.google.com','Google Cloud','cloud gcp infrastructure','Cloud',1900,1300,96),
('https://docker.com','Docker Containers','docker containers devops','DevOps',1600,1000,94),
('https://kubernetes.io','Kubernetes Docs','kubernetes orchestration containers','DevOps',1700,1100,95),
('https://jenkins.io','Jenkins CI CD','jenkins devops automation','DevOps',1400,800,90),
('https://prometheus.io','Prometheus Monitoring','monitoring prometheus metrics','DevOps',1350,750,91),
('https://grafana.com','Grafana Dashboard','grafana monitoring visualization','DevOps',1450,780,92),
('https://react.dev','React Documentation','react frontend javascript','Web',2200,1600,97),
('https://nodejs.org','NodeJS Docs','nodejs backend javascript','Web',2100,1500,96),
('https://expressjs.com','Express Framework','express nodejs backend','Web',1800,1200,94),
('https://developer.mozilla.org','MDN Docs','html css javascript web','Web',2500,1800,98),
('https://postgresql.org','PostgreSQL Database','postgresql sql database','Database',1700,1000,95),
('https://redis.io','Redis Cache','redis cache memory','Database',1650,950,94),
('https://mongodb.com','MongoDB Database','mongodb nosql database','Database',1750,1100,95),
('https://leetcode.com','LeetCode Practice','dsa coding interview','Education',2300,1700,96),
('https://hackerrank.com','HackerRank','coding interview practice','Education',1800,1200,92),
('https://coursera.org','Coursera Courses','online learning courses','Education',2000,1300,93),
('https://udemy.com','Udemy Learning','courses programming learning','Education',1900,1200,91),
('https://edx.org','edX Courses','education online courses','Education',1700,1000,90)
]

cur.executemany(
'INSERT INTO urls(url,title,keywords,category,clicks,backlinks,freshness) VALUES(?,?,?,?,?,?,?)',
urls
)

conn.commit()


## Section 5 - Indexer Service

In [ ]:

df = pd.read_sql('SELECT * FROM urls',conn)

for _,row in df.iterrows():
    for word in row['keywords'].split():
        cur.execute(
            'INSERT INTO search_index(keyword,url_id) VALUES(?,?)',
            (word.lower(),row['id'])
        )

conn.commit()

pd.read_sql('SELECT * FROM search_index LIMIT 20',conn)


## Section 6 - TF-IDF Engine

In [ ]:

documents = df['keywords'].tolist()

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

print("TF-IDF Index Ready")


## Section 7 - Cache with TTL

In [ ]:

cache = {}

CACHE_TTL = 300

cache_hits = 0
cache_misses = 0


## Section 8 - Query Processor

In [ ]:

STOP_WORDS={'the','is','a','an','for','to','of','in','on','best'}

def process_query(query):
    tokens=query.lower().split()
    return [t for t in tokens if t not in STOP_WORDS]


## Section 9 - Analytics Functions

In [ ]:

def log_query(query):
    cur.execute(
        'INSERT INTO query_logs(query,timestamp) VALUES(?,?)',
        (query,datetime.now().isoformat())
    )
    conn.commit()

def click_url(url):

    cur.execute(
        'INSERT INTO click_logs(url,timestamp) VALUES(?,?)',
        (url,datetime.now().isoformat())
    )

    cur.execute(
        'UPDATE urls SET clicks=clicks+1 WHERE url=?',
        (url,)
    )

    conn.commit()


## Section 10 - Search Engine

In [ ]:

def search(query):

    global cache_hits, cache_misses

    print("="*60)
    print("SEARCH REQUEST RECEIVED")
    print("="*60)

    log_query(query)

    if query in cache:

        age = time.time() - cache[query]['timestamp']

        if age < CACHE_TTL:
            cache_hits += 1
            print("CACHE HIT")
            return cache[query]['result']

    cache_misses += 1
    print("CACHE MISS")

    tokens = process_query(query)

    print("Tokens:",tokens)

    query_vector = vectorizer.transform(
        [' '.join(tokens)]
    )

    similarity = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    results=[]

    for idx,score in enumerate(similarity):

        row=df.iloc[idx]

        tfidf_score = score * 100

        click_score = math.log(row['clicks']+1)

        backlink_score = math.log(row['backlinks']+1)

        freshness_score = row['freshness']/10

        final_score=(
            0.50*tfidf_score +
            0.20*click_score +
            0.20*backlink_score +
            0.10*freshness_score
        )

        results.append({
            'Title':row['title'],
            'URL':row['url'],
            'TFIDF':round(tfidf_score,2),
            'Clicks':round(click_score,2),
            'Backlinks':round(backlink_score,2),
            'Freshness':round(freshness_score,2),
            'Final Score':round(final_score,2)
        })

    ranked = pd.DataFrame(results)
    ranked = ranked.sort_values(
        'Final Score',
        ascending=False
    )

    cache[query]={
        'result':ranked,
        'timestamp':time.time()
    }

    return ranked.head(10)

search("python tutorial")


## Section 11 - Click Simulation

In [ ]:

click_url('https://python.org/tutorial')

pd.read_sql(
'SELECT * FROM click_logs ORDER BY id DESC LIMIT 5',
conn
)


## Section 12 - Analytics Dashboard

In [ ]:

top_queries = pd.read_sql('''
SELECT query,
COUNT(*) as searches
FROM query_logs
GROUP BY query
ORDER BY searches DESC
''',conn)

top_queries


## Section 13 - Query Chart

In [ ]:

if len(top_queries)>0:
    plt.figure(figsize=(8,4))
    plt.bar(top_queries['query'],top_queries['searches'])
    plt.title("Top Searches")
    plt.show()


## Section 14 - Monitoring Dashboard

In [ ]:

hit_rate = (
cache_hits/(cache_hits+cache_misses)*100
if (cache_hits+cache_misses)>0
else 0
)

dashboard = pd.DataFrame([
['URLs Indexed',pd.read_sql('SELECT COUNT(*) c FROM urls',conn)['c'][0]],
['Index Entries',pd.read_sql('SELECT COUNT(*) c FROM search_index',conn)['c'][0]],
['Cache Hits',cache_hits],
['Cache Misses',cache_misses],
['Cache Hit Rate %',round(hit_rate,2)],
['Queries Logged',pd.read_sql('SELECT COUNT(*) c FROM query_logs',conn)['c'][0]]
],columns=['Metric','Value'])

dashboard
